# NPU matrix multiplication

This Phase 1C example loads the matching Phase 1B overlay through the repository runtime, tiles logical M/N dimensions over the physical array, and checks every result against NumPy. K must fit the hardware `MAX_K`; the example never bypasses the runtime with direct MMIO or DMA control.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np

repository_root = Path.cwd().resolve()
if not (repository_root / 'src').is_dir():
    repository_root = repository_root.parent
if not (repository_root / 'src').is_dir():
    raise RuntimeError('start this notebook from the repository root or examples directory')
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from src.runtime import TiledMatrixMultiplier, load_pynq_runtime

In [ ]:
artifact_dir = Path('/home/xilinx/jupyter_notebooks/npu_matrix')
manifest_path = artifact_dir / 'npu_matrix.manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
runtime = load_pynq_runtime(artifact_dir / 'npu_matrix.bit')
multiplier = TiledMatrixMultiplier(runtime)
print({
    'source_commit': manifest['source_commit'],
    'vivado_version': manifest['vivado_version'],
    'target_part': manifest['target_part'],
    'physical_limits': (runtime.max_m, runtime.max_n, runtime.max_k),
})

In [ ]:
def reference(a_matrix, b_matrix):
    return (a_matrix.astype(np.int64) @ b_matrix.astype(np.int64)).astype(np.int32)

def run_and_check(label, a_matrix, b_matrix):
    result = multiplier.run(a_matrix, b_matrix, software_timeout=10.0)
    np.testing.assert_array_equal(result.output, reference(a_matrix, b_matrix))
    metrics = result.metrics
    print({
        'case': label,
        'shape': (metrics.m, metrics.k, metrics.n),
        'tile_count': metrics.tile_count,
        'elapsed_seconds': metrics.elapsed_seconds,
        'operation_count': metrics.operation_count,
        'operations_per_second': metrics.operations_per_second,
    })
    return result

In [ ]:
normal_a = np.array([[1, -2, 3], [4, 5, -6]], dtype=np.int8)
normal_b = np.array([[7, 8], [-9, 10], [11, -12]], dtype=np.int8)
normal = run_and_check('normal', normal_a, normal_b)

In [ ]:
non_aligned_a = (np.arange(15).reshape(3, 5) - 7).astype(np.int8)
non_aligned_b = (np.arange(15).reshape(5, 3) - 5).astype(np.int8)
non_aligned = run_and_check('non_aligned', non_aligned_a, non_aligned_b)
assert non_aligned.metrics.tile_count == 4

In [ ]:
repeated = run_and_check('repeated', -non_aligned_a, non_aligned_b)
np.testing.assert_array_equal(repeated.output, reference(-non_aligned_a, non_aligned_b))
print('PASS: NPU matrix multiplication example')